# TensorRT export and benchmark on Kaggle

Attach the Kaggle dataset containing the trained `best.pt` and the BDD100K subset with validation images. Enable a GPU accelerator and Internet. This notebook is GPU-only and writes `benchmark.md`, `benchmark.json`, and `benchmark.png` to `/kaggle/working`.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("TensorRT export requires a CUDA GPU. Enable a Kaggle GPU accelerator and rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
REPO_DIR = Path("/kaggle/working/autonomous-driving-perception-stack")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]", "tensorrt", "onnxruntime-gpu"],
    cwd=REPO_DIR,
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))


In [ ]:
INPUT_DIR = Path("/kaggle/input")
BEST_CANDIDATES = sorted(INPUT_DIR.rglob("best.pt"))
if not BEST_CANDIDATES:
    raise FileNotFoundError("No best.pt found under /kaggle/input; attach the trained model dataset.")
BEST_PT = BEST_CANDIDATES[0]

LABEL_CANDIDATES = sorted(INPUT_DIR.rglob("det_train.json"))
BDD_ROOT = next(
    (
        path.parents[2]
        for path in LABEL_CANDIDATES
        if path.parent.name == "det_20"
        and (path.parents[2] / "images" / "val").is_dir()
    ),
    None,
)
if BDD_ROOT is None:
    raise FileNotFoundError("No BDD100K validation images found under /kaggle/input.")
VAL_IMAGES = BDD_ROOT / "images" / "val"
CALIBRATION_COUNT = 256
if len(list(VAL_IMAGES.glob("*.jpg"))) < CALIBRATION_COUNT:
    raise RuntimeError(f"INT8 calibration requires {CALIBRATION_COUNT} validation JPGs in {VAL_IMAGES}.")
print(f"Weights: {BEST_PT}")
print(f"Calibration images: {VAL_IMAGES}")


In [ ]:
from ultralytics import YOLO

WORK_DIR = Path("/kaggle/working")
MODEL_DIR = WORK_DIR / "models"
ENGINE_DIR = WORK_DIR / "engines"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ENGINE_DIR.mkdir(parents=True, exist_ok=True)
WORKING_MODEL = MODEL_DIR / "best.pt"
if BEST_PT.resolve() != WORKING_MODEL.resolve():
    import shutil

    shutil.copy2(BEST_PT, WORKING_MODEL)
ONNX_PATH = Path(
    YOLO(str(WORKING_MODEL)).export(
        format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False
    )
)
print(f"ONNX model: {ONNX_PATH}")


In [ ]:
for precision, engine_name in (("fp16", "yolo_fp16.engine"), ("int8", "yolo_int8.engine")):
    command = [
        sys.executable,
        "-m",
        "adpstack.export.to_tensorrt",
        "--onnx",
        str(ONNX_PATH),
        "--precision",
        precision,
        "--out",
        str(ENGINE_DIR / engine_name),
        "--imgsz",
        "640",
    ]
    if precision == "int8":
        command.extend(["--calib-dir", str(VAL_IMAGES)])
    subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "adpstack.export.benchmark",
        "--torch-weights",
        str(WORKING_MODEL),
        "--onnx",
        str(ONNX_PATH),
        "--engines",
        f"TensorRT FP16={ENGINE_DIR / 'yolo_fp16.engine'}",
        f"TensorRT INT8={ENGINE_DIR / 'yolo_int8.engine'}",
        "--shape",
        "1",
        "3",
        "640",
        "640",
        "--iters",
        "500",
        "--out",
        str(WORK_DIR / "benchmark"),
    ],
    cwd=REPO_DIR,
    check=True,
)
print("Benchmark files are in /kaggle/working/benchmark.{md,json,png}.")
